> 在线讲义：[第 2 章 · 端侧芯片：看懂 Jetson 的三个数](https://wikieden.github.io/robot-edge-inference/lessons/02-edge-chips.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 2 章 · 端侧芯片：看懂 Jetson 的三个数

配套讲义：仓库里 `ai-infra-edge/notes/lessons/02-edge-chips.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（模组参数来自官方页面；系统预留 3 GB、利用率 40% 是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 留给模型的内存 | 总内存 − 3 | `usable_gb(64)` | 61 GB |
| 第三步 稠密口径 | 稀疏 ÷ 2 | `dense(275)` | 137.5 |
| 第三步 实际算力 | 稠密 × 40% | `effective(dense(275))` | 55 TOPS |
| 第三步 Super Mode | 新 ÷ 旧 | `102 / 68` | 1.50 倍 |
| 第四步 每瓦带宽 | 带宽 ÷ 功率 | `gbps_per_watt(204.8, 60)` | 3.41 GB/s 每瓦 |

模组参数来自 NVIDIA 官方页面（2026-11-26 核对，见 materials/SOURCES.md）。
系统预留 3 GB、实际利用率 40% 是编的，用来演示算法，自己的机器要实测。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
MODULES = {
    "Orin Nano 8GB（原版）": (8, 68, 40, "INT8 稀疏 TOPS", 15),
    "Orin Nano 8GB（Super）": (8, 102, 67, "INT8 稀疏 TOPS", 25),
    "Orin NX 16GB（Super）": (16, 102, 157, "INT8 稀疏 TOPS", 40),
    "AGX Orin 64GB": (64, 204.8, 275, "INT8 稀疏 TOPS", 60),
    "Jetson T5000（Thor）": (128, 273, 2070, "FP4 稀疏 TFLOPS", 130),
}

**函数 `dense`**：官方稀疏口径 ÷ 2 = 稠密口径（大多数模型权重不是 2:4 稀疏的，按稠密算）。

In [ ]:
def dense(sparse: float) -> float:
    """官方稀疏口径 ÷ 2 = 稠密口径（大多数模型权重不是 2:4 稀疏的，按稠密算）。"""
    return sparse / 2

**函数 `usable_gb`**：统一内存里留给模型的部分：总内存减去系统、ROS 2、相机缓冲等预留。

In [ ]:
def usable_gb(total_gb: float, reserve_gb: float = 3.0) -> float:
    """统一内存里留给模型的部分：总内存减去系统、ROS 2、相机缓冲等预留。"""
    return total_gb - reserve_gb

**函数 `effective`**：实际能跑到的算力 ≈ 稠密峰值 × 利用率。

In [ ]:
def effective(dense_peak: float, util: float = 0.4) -> float:
    """实际能跑到的算力 ≈ 稠密峰值 × 利用率。"""
    return dense_peak * util

**函数 `gbps_per_watt`**：每瓦换来多少带宽：端侧最在乎的效率。

In [ ]:
def gbps_per_watt(bw: float, watt: float) -> float:
    """每瓦换来多少带宽：端侧最在乎的效率。"""
    return bw / watt

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：统一内存里真正留给模型的空间（预留 3 GB 是编的）")
for name, (mem, *_rest) in MODULES.items():
    print(f"  {name:<22} {mem:>4} GB → 留给模型约 {usable_gb(mem):>5.0f} GB")

In [ ]:
print("\n第三步：算力的两层折扣（稀疏 → 稠密 → 利用率 40%，40% 是编的）")
name = "AGX Orin 64GB"
sparse = MODULES[name][2]
print(f"  {name}：官方 {sparse} → 稠密 {dense(sparse):.1f} → 实际约 {effective(dense(sparse)):.0f} INT8 TOPS")

In [ ]:
print("\n第三步：Super Mode 是同一块板子提频")
old, new = MODULES["Orin Nano 8GB（原版）"], MODULES["Orin Nano 8GB（Super）"]
print(f"  带宽 {old[1]} → {new[1]} GB/s（×{new[1] / old[1]:.2f}），算力 {old[2]} → {new[2]}（×{new[2] / old[2]:.2f}）")
print(f"  功率上限 {old[4]} → {new[4]} W")

In [ ]:
print("\n第四步：每瓦换来多少带宽（按最高功率算）")
for name, (_m, bw, _t, _u, watt) in MODULES.items():
    print(f"  {name:<22} {bw:>6} GB/s ÷ {watt:>3} W = {gbps_per_watt(bw, watt):.2f} GB/s 每瓦")

## 3. 代码里才看得到的

- `MODULES` 里每一行都带着"算力口径"字段：INT8 稀疏 TOPS 和 FP4 稀疏 TFLOPS 混在一张表里时，不带口径的数字很容易被拿来直接相除。
- 功率用的是官方写的最高功率；实际跑的档位往往更低，每瓦带宽要按你用的档位重算。

## 4. 自己改一改

- 把 `usable_gb` 的预留从 3 GB 改成 5 GB（再开一个大日志进程），看 8 GB 板子还剩多少。
- 把利用率改成 25%，看 AGX Orin 实际只剩多少 TOPS。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 内存大家一块用，先扣系统再算模型；
> 容量带宽加功率，挑板先看这三个；
> 稀疏先除二，再乘利用率；
> 精度不同不能比，带宽口径最老实。